# ავიაციის საიმედოობის ანალიტიკური მონაცემების ტრანსფორმაცია

ეს notebook ახორციელებს ნედლი ფრენების მონაცემების სრულ ტრანსფორმაციას ანალიტიკურ სახეში. პროცესი მოიცავს:
* მონაცემების გასუფთავებას და სტანდარტიზაციას
* KPI მეტრიკების გამოთვლას (დაგვიანების/გაუქმების პროცენტები)
* დაგვიანების მიზეზების ანალიზს 5 კატეგორიის მიხედვით
* ოპერაციული რისკის შეფასებას
* დუბლიკატების მოცილებას

საბოლოო პროდუქტი: **getdata.calculated.AviationReliability** - ცხრილი რომელიც მზადაა დაშბორდებისა და ვიზუალიზაციისთვის.


In [0]:
CREATE SCHEMA IF NOT EXISTS getdata.calculated;

## მთავარი ტრანსფორმაციის ლოგიკა

მრავალსაფეხურიანი მონაცემთა დამუშავება 3 CTE-ის საშუალებით:

**1. CleanedRawData** - ასუფთავებს და აერთიანებს:
* ფაქტების ცხრილს (airline_airport_monthly)
* ავიაკომპანიების რანკინგებს
* აეროპორტების რანკინგებს

**2. CalculatedMetrics** - ითვლის:
* დაგვიანების/გაუქმების პროცენტებს
* საშუალო დაგვიანების ხანგრძლივობას
* თითოეული მიზეზის წილს ჯამურ დაგვიანებაში

**3. CalculatedFinal** - ამატებს:
* თვიურ აგრეგაციებს (Window ფუნქციები)
* ოპერაციული რისკის კატეგორიებს
* პირველადი დაგვიანების მიზეზის იდენტიფიკაციას
* დუბლიკატების ფილტრაციას (QUALIFY)


In [0]:
CREATE OR REPLACE TABLE getdata.calculated.AviationReliability AS

WITH CleanedRawData AS (
    SELECT 
        -- დროითი განზომილებები
        CAST(Mnth.year AS INT) AS FlightYear, -- წელი გარდაიქმნება მთელ რიცხვად (INT) სორტირებისა და ფილტრაციისთვის.
        CAST(Mnth.month AS INT) AS FlightMonth, -- თვე გარდაიქმნება მთელ რიცხვად (INT) დროითი ანალიზისთვის.
        TRIM(Mnth.period) AS Period, -- ტექსტური ფორმატი (YYYY-MM) სუფთავდება ზედმეტი ჰარებისგან (Spaces).

        --ავიაკომპანიისა და აეროპორტის იდენტიფიკატორები
        TRIM(Mnth.carrier) AS CarrierCode,
        TRIM(Mnth.carrier_name) AS CarrierName, -- ავიაკომპანიის ოფიციალური დასახელება.
        TRIM(Mnth.airport) AS AirportCode, 
        TRIM(Mnth.airport_name) AS AirportName, -- აეროპორტისა და ქალაქის სრული დასახელება.

        -- ფრენების რაოდენობრივი მეტრიკები (BIGINT და DOUBLE ტიპებში გადაყვანა,ასევე NULL-ების 0-ით ჩანაცვლება)
        COALESCE(CAST(Mnth.arr_flights AS BIGINT), 0) AS TotalRepresentedFlights,
        COALESCE(CAST(Mnth.arr_del15 AS BIGINT), 0) AS DelayedFlights15Plus, 
        COALESCE(CAST(Mnth.arr_cancelled AS BIGINT), 0) AS CancelledFlights,
        COALESCE(CAST(Mnth.arr_diverted AS BIGINT), 0) AS DivertedFlights, 
        COALESCE(CAST(Mnth.arr_delay AS DOUBLE), 0.0) AS TotalDelayMinutes, -- დაგვიანების წუთები გადადის DOUBLE ტიპში (ათწილადი) მათემატიკური გამოთვლებისთვის.

        -- დაგვიანების წუთები 5 მიზეზის მიხედვით (DOUBLE ტიპი + NULL-ების 0.0-ით ჩანაცვლება)
        COALESCE(CAST(Mnth.carrier_delay AS DOUBLE), 0.0) AS CarrierDelayMinutes, -- ავიაკომპანიის მიზეზით გამოწვეული დაგვიანების წუთები.
        COALESCE(CAST(Mnth.weather_delay AS DOUBLE), 0.0) AS WeatherDelayMinutes, -- ამინდის მიზეზით გამოწვეული დაგვიანების წუთები.
        COALESCE(CAST(Mnth.nas_delay AS DOUBLE), 0.0) AS NasDelayMinutes, -- აეროპორტის/NAS-ის მიზეზით გამოწვეული დაგვიანების წუთები.
        COALESCE(CAST(Mnth.security_delay AS DOUBLE), 0.0) AS SecurityDelayMinutes, -- უსაფრთხოების შემოწმებით გამოწვეული დაგვიანების წუთები.
        COALESCE(CAST(Mnth.late_aircraft_delay AS DOUBLE), 0.0) AS LateAircraftDelayMinutes, -- წინა რეისის დაგვიანებით გამოწვეული დაგვიანების წუთები.

        --ოფიციალური რანგები ცნობარებიდან
        AlR.delay_rate_rank AS AirlineDelayRank, 
        ApR.delay_rate_rank AS AirportDelayRank

    FROM getdata.raw.airline_airport_monthly AS Mnth 
    -- LEFT JOIN ინარჩუნებს ფაქტების ცხრილის ყველა მწკრივს, მაშინაც კი თუ ცნობარში კოდი არ მოიძებნა.
    LEFT JOIN getdata.raw.airline_rankings_2025_2026 AS AlR 
        ON Mnth.carrier = AlR.carrier
    LEFT JOIN getdata.raw.airport_rankings_2025_2026 AS ApR 
        ON Mnth.airport = ApR.airport
),


--  CalculatedMetrics (CTE)
-- პასუხისმგებელია პროცენტული მაჩვენებლებისა და საშუალოების დათვლაზე:
-- CASE WHEN TotalRepresentedFlights = 0 THEN 0.0 -> იცავს კოდს Division by Zero შეცდომისგან.
--  CAST(... AS DOUBLE) -> უზრუნველყოფს ათწილადურ გაყოფას.
-- ROUND(..., 2) -> ამრგვალებს შედეგებს 2 ათწილადამდე.

CalculatedMetrics AS (
    SELECT 
        C.*, -- შემოაქვს CleanedRawData CTE-ის ყველა გასუფთავებული სვეტი.

        --დაგვიანებული ფრენების პროცენტული წილი (% = (DelayedFlights15Plus / TotalRepresentedFlights) * 100)
        ROUND(
            CASE 
                WHEN C.TotalRepresentedFlights = 0 THEN 0.0 
                ELSE (CAST(C.DelayedFlights15Plus AS DOUBLE) / CAST(C.TotalRepresentedFlights AS DOUBLE)) * 100.0 -- DOUBLE  ზუსტი პროცენტის მისაღებად.
            END, 2
        ) AS DelayRatePct,

        --გაუქმებული ფრენების პროცენტული წილი (% = (CancelledFlights / TotalRepresentedFlights) * 100)
        ROUND(
            CASE 
                WHEN C.TotalRepresentedFlights = 0 THEN 0.0 
                ELSE (CAST(C.CancelledFlights AS DOUBLE) / CAST(C.TotalRepresentedFlights AS DOUBLE)) * 100.0 
            END, 2
        ) AS CancellationRatePct,

        --საშუალო დაგვიანების ხანგრძლივობა ერთ დაგვიანებულ ფრენაზე (= TotalDelayMinutes / DelayedFlights15Plus)
        ROUND(
            CASE 
                WHEN C.DelayedFlights15Plus = 0 THEN 0.0 
                ELSE C.TotalDelayMinutes / CAST(C.DelayedFlights15Plus AS DOUBLE) 
            END, 2
        ) AS AvgDelayMinutesPerDelayed,

        --ავიაკომპანიის ბრალეულობის წილი დაგვიანებულ წუთებში (% = (CarrierDelayMinutes / TotalDelayMinutes) * 100)
        ROUND(
            CASE 
                WHEN C.TotalDelayMinutes = 0.0 THEN 0.0 
                ELSE (C.CarrierDelayMinutes / C.TotalDelayMinutes) * 100.0 
            END, 2
        ) AS CarrierDelaySharePct,

        --ამინდის ბრალეულობის წილი დაგვიანებულ წუთებში (% = (WeatherDelayMinutes / TotalDelayMinutes) * 100)
        ROUND(
            CASE 
                WHEN C.TotalDelayMinutes = 0.0 THEN 0.0 
                ELSE (C.WeatherDelayMinutes / C.TotalDelayMinutes) * 100.0 
            END, 2
        ) AS WeatherDelaySharePct,

        --აეროპორტის/NAS-ის ბრალეულობის წილი დაგვიანებულ წუთებში (% = (NasDelayMinutes / TotalDelayMinutes) * 100)
        ROUND(
            CASE 
                WHEN C.TotalDelayMinutes = 0.0 THEN 0.0 
                ELSE (C.NasDelayMinutes / C.TotalDelayMinutes) * 100.0 
            END, 2
        ) AS NasDelaySharePct,

        -- უსაფრთხოების ბრალეულობის წილი დაგვიანებულ წუთებში (% = (SecurityDelayMinutes / TotalDelayMinutes) * 100)
        ROUND(
            CASE 
                WHEN C.TotalDelayMinutes = 0.0 THEN 0.0 -- თუ დაგვიანების წუთები 0-ია, იწერება 0.0 (Division by Zero დაცვა).
                ELSE (C.SecurityDelayMinutes / C.TotalDelayMinutes) * 100.0 -- უსაფრთხოების შემოწმების წილი დაგვიანების ჯამურ წუთებში.
            END, 2
        ) AS SecurityDelaySharePct,

        -- წინა რეისის დაგვიანების წილი დაგვიანებულ წუთებში (% = (LateAircraftDelayMinutes / TotalDelayMinutes) * 100)
        ROUND(
            CASE 
                WHEN C.TotalDelayMinutes = 0.0 THEN 0.0 
                ELSE (C.LateAircraftDelayMinutes / C.TotalDelayMinutes) * 100.0 
            END, 2
        ) AS LateAircraftDelaySharePct

    FROM CleanedRawData AS C -- იყენებს პირველი CTE-ის გასუფთავებულ მონაცემებს.
),

-- CalculatedFinal (CTE)
-- პასუხისმგებელია სტრუქტურიზაციაზე, Window ფუნქციებსა და ბიზნეს-კატეგორიზაციაზე:
-- ROW_NUMBER() -> პირველადი გასაღების (AnalyticsKey) გენერირება.
--  SUM(...) OVER (...) -> თვიური კონტექსტური აგრეგაციები Window ფუნქციით.
--  CASE WHEN -> ბიზნეს-სეგმენტაცია (ოპერაციული რისკი + მთავარი მიზეზი).

CalculatedFinal AS (
    SELECT 
        
        ROW_NUMBER() OVER (
            ORDER BY M.Period, M.CarrierCode, M.AirportCode
            ) AS AnalyticsKey, -- აგენერირებს უნიკალურ ID-ს ყოველი მწკრივისთვის, დალაგებულს პერიოდის, ავიაკომპანიისა და აეროპორტის მიხედვით.
        M.FlightYear,
        M.FlightMonth,
        M.Period,
        M.CarrierCode,
        M.CarrierName,
        M.AirportCode,
        M.AirportName,
        -- რაოდენობრივი მეტრიკები
        M.TotalRepresentedFlights,
        M.DelayedFlights15Plus,
        M.CancelledFlights,
        M.DivertedFlights,
        M.TotalDelayMinutes,
        -- დაგვიანების წუთები მიზეზების მიხედვით
        M.CarrierDelayMinutes,
        M.WeatherDelayMinutes,
        M.NasDelayMinutes,
        M.SecurityDelayMinutes,
        M.LateAircraftDelayMinutes,
        -- გამოთვლილი პროცენტები და საშუალოები
        M.DelayRatePct,
        M.CancellationRatePct,
        M.AvgDelayMinutesPerDelayed,
        -- მიზეზების პროცენტული წილები
        M.CarrierDelaySharePct,
        M.WeatherDelaySharePct,
        M.NasDelaySharePct,
        M.SecurityDelaySharePct,
        M.LateAircraftDelaySharePct,
        M.AirlineDelayRank,
        M.AirportDelayRank,
        -- Window აგრეგაციები
    SUM(M.TotalRepresentedFlights) OVER (
        PARTITION BY M.CarrierCode, M.Period
        ) AS TotalCarrierMonthlyFlights, -- ითვლის ავიაკომპანიის ჯამურ ფრენებს აშშ-ის მასშტაბით მოცემულ თვეში.
    SUM(M.TotalRepresentedFlights) OVER (
        PARTITION BY M.AirportCode, M.Period
        ) AS TotalAirportMonthlyFlights, -- ითვლის აეროპორტის ჯამურ ფრენებს ყველა ავიაკომპანიის მიერ მოცემულ თვეში.

        -- ოპერაციული რისკი
    CASE 
            WHEN M.CancellationRatePct >= 5.0 OR M.DelayRatePct >= 25.0 THEN 'კრიტიკული რისკი' 
            WHEN M.DelayRatePct >= 18.0 OR M.CancellationRatePct >= 2.5 THEN 'მაღალი რისკი' 
            WHEN M.DelayRatePct >= 10.0 THEN 'საშუალო რისკი' 
            ELSE 'სტაბილური' 
    END AS OperationalRiskStatus, 

        -- ბიზნეს-კატეგორიზაცია: დომინანტი მიზეზი
    CASE 
            WHEN M.LateAircraftDelaySharePct >= 35.0 THEN 'წინა რეისის დაგვიანება'
            WHEN M.CarrierDelaySharePct >= 35.0 THEN 'ავიაკომპანიის ხარვეზი'
            WHEN M.NasDelaySharePct >= 35.0 THEN 'აეროპორტი / NAS'
            WHEN M.WeatherDelaySharePct >= 20.0 THEN 'ცუდი ამინდი'
            ELSE 'მრავალფაქტორიანი' 
    END AS PrimaryDelayReason

    FROM CalculatedMetrics AS M -- იყენებს მეორე CTE-ის გამოთვლილ მეტრიკებს.
)

-- საბოლოო SELECT ბლოკი QUALIFY ფილტრით
-- QUALIFY + ROW_NUMBER() უზრუნველყოფს დუბლიკატების მოსპობას.
-- თითოეულ კომბინაციაზე (Period + CarrierCode + AirportCode) ტოვებს მხოლოდ 1 მწკრივს 
-- ყველაზე მაღალი ფრენების რაოდენობით (ORDER BY TotalRepresentedFlights DESC).
SELECT * FROM CalculatedFinal
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY Period, CarrierCode, AirportCode 
    ORDER BY TotalRepresentedFlights DESC
) = 1;


##  SELECT COUNT(*) 

ამოწმებს, რომ ბაზაში ჩაიწერა ზუსტად 32,013 გასუფთავებული მწკრივი.

In [0]:
SELECT COUNT(*) AS TotalCalculatedRows 
FROM getdata.calculated.AviationReliability;

## მონაცემების ხარისხის შემოწმება

აჩვენებს პირველ 10 მწკრივს შექმნილი ცხრილიდან. 


In [0]:
SELECT * 
FROM getdata.calculated.AviationReliability
LIMIT 10;